# Baseline 1: Direct LLM (no verification)

**Request → Qwen2.5-3B-Instruct → code → sandbox → output.** Every transformation that runs is accepted.

This notebook also produces **candidate pool v1** (`candidates.jsonl`): the LLM code that Baselines 2–3 and the proposed system will judge later.

**Setup on Kaggle**
1. Upload the project as a Kaggle Dataset containing at least `data/*.csv` and `eval/tasks/**/*.yaml` (e.g. zip the repo folder). Add it to this notebook with *Add Input*.
2. *Settings → Accelerator:* **GPU T4 x1** (or P100). *Settings → Internet:* **on** (pip install + model download).
3. First run with `SPLITS = ["dev"]` (or `LIMIT_TASKS = 3`) to check the prompt; then set `SPLITS = ["dev", "test"]` and run all.

**Pinned data stack:** pandas 2.3.3, numpy 2.1.3. They are installed into a separate folder and used only by the sandbox subprocesses, so Kaggle's torch/transformers stay untouched. All schema extraction, reference runs, LLM-code runs and comparisons happen inside that pinned environment.

Outputs go to `/kaggle/working/baseline1/` and are zipped at the end.

In [1]:
# ---------------- Configuration ----------------
import ast, glob, hashlib, json, math, os, re, shutil, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
SPLITS = ["dev","test"]      # first run: ["dev"] to check the prompt
LIMIT_TASKS = None            # e.g. 3 for a quick dry run
N_SAMPLES = 5                 # sampled generations per task (0 = greedy only)
TEMPERATURE = 0.7
TOP_P = 0.95
MAX_NEW_TOKENS = 512
GREEDY_BATCH = 8
SEED = 1234

EXEC_TIMEOUT_S = 60           # wall-clock limit per candidate (subprocess killed after this)
CPU_LIMIT_S = 30              # CPU-time limit inside the sandbox
MEM_LIMIT_GB = 4              # address-space limit inside the sandbox
EVAL_WORKERS = 4              # candidates evaluated in parallel

PINNED = {"pandas": "2.3.3", "numpy": "2.1.3", "pyyaml": "6.0.2"}
PROMPT_VERSION = "b1-v1"

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "kaggle_working"
OUT = WORK / "baseline1"
OUT.mkdir(parents=True, exist_ok=True)


def find_root():
    """Folder that contains data/ and eval/tasks/ (Kaggle input, or IGT_ROOT for local runs)."""
    cands = []
    if os.environ.get("IGT_ROOT"):
        cands.append(Path(os.environ["IGT_ROOT"]))
    cands += [Path(p).parent.parent for p in glob.glob("/kaggle/input/**/eval/tasks", recursive=True)] 
    cands += [Path.cwd(), Path.cwd().parent]
    for c in cands:
        if (c / "eval" / "tasks").is_dir() and (c / "data").is_dir():
            return c.resolve()
    raise FileNotFoundError("Could not find a folder with data/ and eval/tasks/. Add the project dataset as input.")


ROOT = find_root()
print("project root:", ROOT)
print("outputs     :", OUT)

project root: /kaggle/input/datasets/srinivasu2005/lma-eval-dataset
outputs     : /kaggle/working/baseline1


In [2]:
# ---------------- Pinned pandas/numpy for the sandbox ----------------
# Installed into a separate folder and put first on PYTHONPATH for sandbox subprocesses only.
EVALPKGS = WORK / "evalpkgs"
EVAL_PYTHON = os.environ.get("IGT_EVAL_PYTHON")   # local runs: point to a venv python that already has the pins

if EVAL_PYTHON:
    EVAL_ENV = dict(os.environ)
else:
    EVAL_PYTHON = sys.executable
    if not (EVALPKGS / "pandas").exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--target", str(EVALPKGS),
                        *[f"{k}=={v}" for k, v in PINNED.items()]], check=True)
    EVAL_ENV = dict(os.environ, PYTHONPATH=str(EVALPKGS), PYTHONNOUSERSITE="1")

probe = subprocess.run([EVAL_PYTHON, "-c", "import pandas, numpy, yaml; print(pandas.__version__, numpy.__version__, yaml.__version__)"],
                       env=EVAL_ENV, capture_output=True, text=True, check=True).stdout.split()
EVAL_VERSIONS = dict(zip(["pandas", "numpy", "pyyaml"], probe))
print("sandbox versions:", EVAL_VERSIONS)
assert EVAL_VERSIONS["pandas"] == PINNED["pandas"] and EVAL_VERSIONS["numpy"] == PINNED["numpy"], EVAL_VERSIONS

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
aiobotocore 2.26.0 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
botocore 1.41.5 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
holidays 0.103 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 3.0.0 which is 

sandbox versions: {'pandas': '2.3.3', 'numpy': '2.1.3', 'pyyaml': '6.0.2'}


In [3]:
# ---------------- Sandbox harness (runs inside the pinned environment) ----------------
HARNESS = r'''
import contextlib, glob, io, json, os, pickle, resource, sys, time
import numpy as np
import pandas as pd
import yaml


def load_tasks(root):
    tasks = []
    for f in sorted(glob.glob(f"{root}/eval/tasks/*/*.yaml")):
        with open(f) as fh:
            tasks.append(yaml.safe_load(fh))
    return tasks


def load_data(root, data_file):
    return pd.read_csv(f"{root}/{data_file}")


def family(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype):
        return "category"
    if pd.api.types.is_bool_dtype(dt):
        return "bool"
    if pd.api.types.is_integer_dtype(dt):
        return "int"
    if pd.api.types.is_float_dtype(dt):
        return "float"
    if pd.api.types.is_datetime64_any_dtype(dt):
        return "datetime"
    if pd.api.types.is_object_dtype(dt):
        return "string" if all(isinstance(v, str) for v in s.dropna()) else "object"
    if pd.api.types.is_string_dtype(dt):
        return "string"
    return str(dt)


def norm(v):
    if v is None:
        return ("N",)
    try:
        if pd.isna(v):
            return ("N",)
    except (TypeError, ValueError):
        pass
    if isinstance(v, (bool, np.bool_)):
        return ("B", bool(v))
    if isinstance(v, (int, float, np.integer, np.floating)):
        return ("X", round(float(v), 9))
    if isinstance(v, (pd.Timestamp, np.datetime64)):
        return ("T", pd.Timestamp(v).isoformat())
    return ("S", str(v))


def outputs_equal(ref, cand):
    """Answer key: content comparison. Row order, index and column order are ignored."""
    if not isinstance(cand, pd.DataFrame):
        return False, "not a DataFrame"
    if cand.columns.duplicated().any():
        return False, "duplicate column names"
    if set(ref.columns) != set(cand.columns):
        extra = sorted(map(str, set(cand.columns) - set(ref.columns)))
        missing = sorted(map(str, set(ref.columns) - set(cand.columns)))
        return False, f"columns differ (extra={extra}, missing={missing})"
    if len(ref) != len(cand):
        return False, f"row count differs ({len(cand)} vs {len(ref)})"
    cols = list(ref.columns)
    for c in cols:
        fr, fc = family(ref[c]), family(cand[c])
        if fr != fc:
            return False, f"dtype family differs in {c} ({fc} vs {fr})"
        if fr == "category" and ref[c].cat.ordered:
            if not cand[c].cat.ordered or list(ref[c].cat.categories) != list(cand[c].cat.categories):
                return False, f"category order differs in {c}"

    def rows(df):
        return sorted(zip(*[[norm(v) for v in df[c].astype(object).tolist()] for c in cols]))

    if rows(ref) != rows(cand):
        return False, "values differ"
    return True, "equal"


def index_kept(before, out):
    """Informational: is the output index still usable as the row id?"""
    try:
        if not out.index.is_unique or not out.index.isin(before.index).all():
            return False
        common = [c for c in before.columns if c in out.columns]
        if not common or len(out) == 0:
            return True
        bv = before.loc[out.index, common].astype(object).values
        ov = out[common].astype(object).values
        agree = sum(2 * sum(norm(x) == norm(y) for x, y in zip(b, o)) >= len(common) for b, o in zip(bv, ov))
        return agree / len(out) >= 0.95
    except Exception:
        return False


def exec_transform(code, func_name, df):
    ns = {"pd": pd, "np": np, "__name__": "candidate"}
    sink = io.StringIO()
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        exec(compile(code, "<candidate>", "exec"), ns)
        fn = ns.get(func_name)
        if not callable(fn):
            raise NameError(f"function '{func_name}' is not defined")
        return fn(df.copy())


def set_limits(cpu_s, mem_gb):
    resource.setrlimit(resource.RLIMIT_CPU, (cpu_s, cpu_s + 1))
    lim = int(mem_gb * 1024 ** 3)
    resource.setrlimit(resource.RLIMIT_AS, (lim, lim))


def cmd_schema(root, out_path):
    tasks = load_tasks(root)
    datasets = {}
    for t in tasks:
        f = t["data_file"]
        if f in datasets:
            continue
        df = load_data(root, f)
        datasets[f] = dict(
            n_rows=len(df),
            columns=[dict(name=str(c), dtype=str(df[c].dtype), nulls=int(df[c].isna().sum())) for c in df.columns],
            head_csv=df.head(5).to_csv(index=False),
        )
    slim = [dict(id=t["id"], split=t["split"], dataset=t["dataset"], transformation_type=t["transformation_type"],
                 difficulty=t.get("difficulty"), request=" ".join(str(t["request"]).split()), data_file=t["data_file"])
            for t in tasks]
    with open(out_path, "w") as fh:
        json.dump(dict(tasks=slim, datasets=datasets), fh)


def cmd_refs(root, ref_dir, out_path):
    os.makedirs(ref_dir, exist_ok=True)
    report = []
    for t in load_tasks(root):
        df = load_data(root, t["data_file"])
        rec = dict(id=t["id"])
        try:
            out = exec_transform(t["reference_code"], "transform", df)
            with open(f"{ref_dir}/{t['id']}.pkl", "wb") as fh:
                pickle.dump(out, fh)
            rec.update(ok=True, rows_after=len(out), expected_rows_after=t["expected_effect"]["rows_after"],
                       self_equal=outputs_equal(out, out)[0], noop_equal=outputs_equal(out, df)[0])
        except Exception as e:
            rec.update(ok=False, error=f"{type(e).__name__}: {e}")
        report.append(rec)
    with open(out_path, "w") as fh:
        json.dump(report, fh)


def cmd_run(root, ref_dir, job_path, out_path, cpu_s, mem_gb):
    with open(job_path) as fh:
        job = json.load(fh)
    df = load_data(root, job["data_file"])
    with open(f"{ref_dir}/{job['task_id']}.pkl", "rb") as fh:
        ref = pickle.load(fh)
    res = dict(status="error", ran_ok=False, correct=False, rows_before=len(df))
    try:
        set_limits(int(cpu_s), float(mem_gb))
    except Exception:
        pass
    t0 = time.time()
    try:
        out = exec_transform(job["code"], job["func_name"], df)
        res["runtime_s"] = round(time.time() - t0, 3)
        if not isinstance(out, pd.DataFrame):
            res.update(status="bad_return", error=f"returned {type(out).__name__}, not a DataFrame")
        else:
            ok, reason = outputs_equal(ref, out)
            res.update(status="ok", ran_ok=True, correct=bool(ok), reason=reason, rows_after=len(out),
                       cols_added=sorted(map(str, set(out.columns) - set(df.columns))),
                       cols_removed=sorted(map(str, set(df.columns) - set(out.columns))),
                       index_kept=bool(index_kept(df, out)))
    except MemoryError:
        res.update(status="memory", error="MemoryError")
    except BaseException as e:
        res.update(status="error", error=f"{type(e).__name__}: {str(e)[:300]}")
    with open(out_path, "w") as fh:
        json.dump(res, fh, default=str)


if __name__ == "__main__":
    cmd, args = sys.argv[1], sys.argv[2:]
    {"schema": cmd_schema, "refs": cmd_refs, "run": cmd_run}[cmd](*args)
'''
HARNESS_PATH = OUT / "harness.py"
HARNESS_PATH.write_text(HARNESS)
REF_DIR = OUT / "refs"


def harness(*args, timeout=None, cwd=None):
    return subprocess.run([EVAL_PYTHON, str(HARNESS_PATH), *map(str, args)], env=EVAL_ENV,
                          capture_output=True, text=True, timeout=timeout, cwd=cwd)


print("harness written to", HARNESS_PATH)

harness written to /kaggle/working/baseline1/harness.py


In [4]:
# ---------------- Load tasks and schemas (inside the pinned environment) ----------------
p = harness("schema", ROOT, OUT / "schema.json")
assert p.returncode == 0, p.stderr
SCHEMA = json.loads((OUT / "schema.json").read_text())
ALL_TASKS = SCHEMA["tasks"]
TASK_INDEX = {t["id"]: i for i, t in enumerate(ALL_TASKS)}        # stable seeds regardless of SPLITS
TASKS = [t for t in ALL_TASKS if t["split"] in SPLITS]
if LIMIT_TASKS:
    TASKS = TASKS[:LIMIT_TASKS]
print(f"{len(ALL_TASKS)} tasks found, {len(TASKS)} selected (splits={SPLITS}, limit={LIMIT_TASKS})")

60 tasks found, 60 selected (splits=['dev', 'test'], limit=None)


In [5]:
# ---------------- Sanity check: references (answer key) ----------------
p = harness("refs", ROOT, REF_DIR, OUT / "refs_report.json")
assert p.returncode == 0, p.stderr
report = json.loads((OUT / "refs_report.json").read_text())
problems = [r for r in report if not r["ok"] or r["rows_after"] != r["expected_rows_after"]
            or not r["self_equal"] or r["noop_equal"]]
print(f"references run: {sum(r['ok'] for r in report)}/{len(report)}")
print("reference == itself for all tasks:", all(r.get("self_equal") for r in report))
print("no-op (return df) is labelled wrong for all tasks:", not any(r.get("noop_equal") for r in report))
for r in problems:
    print("PROBLEM:", r)
assert not problems, "Fix the task files before generating."

references run: 60/60
reference == itself for all tasks: True
no-op (return df) is labelled wrong for all tasks: True


In [6]:
# ---------------- Prompt (identical for every system later) ----------------
SYSTEM_PROMPT = """You are an expert Python data engineer. Write pandas code that performs exactly the transformation the user asks for.

Rules:
- Define a function `transform(df)` that takes a pandas DataFrame and returns the transformed DataFrame.
- `df` is already loaded and is passed to the function. Do not read or write files and do not use the network.
- pandas (as pd) and numpy (as np) are available.
- Do not print anything.
- Reply with a single ```python code block containing the function, and nothing else."""

USER_TEMPLATE = """Dataset: {n_rows} rows, {n_cols} columns.

Columns (name: dtype, missing values):
{columns}

First 5 rows (CSV):
{head}
Request: {request}"""

PROMPT_HASH = hashlib.sha256((PROMPT_VERSION + SYSTEM_PROMPT + USER_TEMPLATE).encode()).hexdigest()[:12]


def build_messages(task):
    ds = SCHEMA["datasets"][task["data_file"]]
    cols = "\n".join(f"- {c['name']}: {c['dtype']}, {c['nulls']} missing" for c in ds["columns"])
    user = USER_TEMPLATE.format(n_rows=ds["n_rows"], n_cols=len(ds["columns"]), columns=cols,
                                head=ds["head_csv"], request=task["request"])
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]


print("prompt version:", PROMPT_VERSION, "hash:", PROMPT_HASH, "\n")
print(build_messages(TASKS[0])[1]["content"])

prompt version: b1-v1 hash: ebbe8bceae71 

Dataset: 1000 rows, 15 columns.

Columns (name: dtype, missing values):
- age: int64, 0 missing
- workclass: object, 62 missing
- fnlwgt: int64, 0 missing
- education: object, 0 missing
- education-num: int64, 0 missing
- marital-status: object, 0 missing
- occupation: object, 62 missing
- relationship: object, 0 missing
- race: object, 0 missing
- sex: object, 0 missing
- capital-gain: int64, 0 missing
- capital-loss: int64, 0 missing
- hours-per-week: int64, 0 missing
- native-country: object, 18 missing
- income: object, 0 missing

First 5 rows (CSV):
age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
38,Pri

In [7]:
# ---------------- Load the model ----------------
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("model loaded on", model.device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


def render(messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.inference_mode()
def generate(prompts, do_sample, n=1, seed=None):
    """Returns (texts, prompt_token_counts, output_token_counts); n outputs per prompt when sampling."""
    if seed is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id)
    if do_sample:
        kw.update(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, num_return_sequences=n)
    else:
        kw.update(do_sample=False, temperature=None, top_p=None, top_k=None)
    out = model.generate(**enc, **kw)
    gen = out[:, enc["input_ids"].shape[1]:]
    texts = tok.batch_decode(gen, skip_special_tokens=True)
    prompt_tokens = enc["attention_mask"].sum(dim=1).tolist()
    prompt_tokens = [pt for pt in prompt_tokens for _ in range(n if do_sample else 1)]
    out_tokens = [int((g != tok.pad_token_id).sum()) for g in gen]
    return texts, prompt_tokens, out_tokens

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model loaded on cuda:0 | GPU: Tesla T4


In [8]:
# ---------------- Generate (greedy + sampled), resumable ----------------
GEN_PATH = OUT / "generations.jsonl"


def load_jsonl(path):
    return [json.loads(l) for l in open(path)] if Path(path).exists() else []


def append_jsonl(path, records):
    with open(path, "a") as fh:
        for r in records:
            fh.write(json.dumps(r) + "\n")


def gen_record(task, mode, k, seed, temp, text, pt, ot, secs):
    return dict(gen_id=f"{task['id']}__{mode}{k}", task_id=task["id"], split=task["split"], dataset=task["dataset"],
                transformation_type=task["transformation_type"], difficulty=task["difficulty"], mode=mode, sample_idx=k,
                seed=seed, temperature=temp, top_p=TOP_P if mode == "sample" else None, model=MODEL_ID,
                prompt_version=PROMPT_VERSION, prompt_hash=PROMPT_HASH, max_new_tokens=MAX_NEW_TOKENS,
                raw_output=text, prompt_tokens=pt, output_tokens=ot, truncated=ot >= MAX_NEW_TOKENS,
                gen_time_s=round(secs, 2))


done = {r["gen_id"] for r in load_jsonl(GEN_PATH)}
t_start = time.time()

greedy_todo = [t for t in TASKS if f"{t['id']}__greedy0" not in done]
for i in range(0, len(greedy_todo), GREEDY_BATCH):
    batch = greedy_todo[i:i + GREEDY_BATCH]
    t0 = time.time()
    texts, pts, ots = generate([render(build_messages(t)) for t in batch], do_sample=False)
    secs = (time.time() - t0) / len(batch)
    append_jsonl(GEN_PATH, [gen_record(t, "greedy", 0, None, 0.0, x, pt, ot, secs) for t, x, pt, ot in zip(batch, texts, pts, ots)])
    print(f"greedy {i + len(batch)}/{len(greedy_todo)}  ({time.time() - t_start:.0f}s)")

for j, t in enumerate(TASKS):
    if N_SAMPLES == 0:
        break
    if all(f"{t['id']}__sample{k}" in done for k in range(N_SAMPLES)):
        continue
    seed = SEED + 1000 + TASK_INDEX[t["id"]]
    t0 = time.time()
    texts, pts, ots = generate([render(build_messages(t))], do_sample=True, n=N_SAMPLES, seed=seed)
    secs = (time.time() - t0) / N_SAMPLES
    append_jsonl(GEN_PATH, [gen_record(t, "sample", k, seed, TEMPERATURE, x, pt, ot, secs)
                            for k, (x, pt, ot) in enumerate(zip(texts, pts, ots))])
    print(f"sampled {j + 1}/{len(TASKS)}  ({time.time() - t_start:.0f}s)")

print("generations on disk:", len(load_jsonl(GEN_PATH)))

greedy 8/60  (6s)
greedy 16/60  (11s)
greedy 24/60  (15s)
greedy 32/60  (19s)
greedy 40/60  (25s)
greedy 48/60  (30s)
greedy 56/60  (35s)
greedy 60/60  (39s)
sampled 1/60  (43s)
sampled 2/60  (45s)
sampled 3/60  (49s)
sampled 4/60  (52s)
sampled 5/60  (57s)
sampled 6/60  (60s)
sampled 7/60  (64s)
sampled 8/60  (68s)
sampled 9/60  (75s)
sampled 10/60  (78s)
sampled 11/60  (81s)
sampled 12/60  (87s)
sampled 13/60  (91s)
sampled 14/60  (97s)
sampled 15/60  (102s)
sampled 16/60  (106s)
sampled 17/60  (111s)
sampled 18/60  (117s)
sampled 19/60  (120s)
sampled 20/60  (123s)
sampled 21/60  (126s)
sampled 22/60  (133s)
sampled 23/60  (138s)
sampled 24/60  (143s)
sampled 25/60  (146s)
sampled 26/60  (149s)
sampled 27/60  (151s)
sampled 28/60  (155s)
sampled 29/60  (159s)
sampled 30/60  (162s)
sampled 31/60  (166s)
sampled 32/60  (169s)
sampled 33/60  (175s)
sampled 34/60  (179s)
sampled 35/60  (185s)
sampled 36/60  (189s)
sampled 37/60  (193s)
sampled 38/60  (198s)
sampled 39/60  (202s)
sampled

In [9]:
# ---------------- Code extraction and AST safety check ----------------
FENCE_RE = re.compile(r"```[ \t]*(?:python|py|Python)?[ \t]*\n(.*?)```", re.S)


def candidate_blocks(text):
    blocks = FENCE_RE.findall(text)
    if "```" in text:                                   # unterminated block (e.g. output truncated)
        tail = text.split("```", 1)[1]
        tail = tail.split("\n", 1)[1] if "\n" in tail else ""
        if "```" not in tail:
            blocks.append(tail)
    blocks.append(text)                                 # unfenced reply
    return blocks


def keep_definitions(tree):
    """Keep imports, function/class definitions and literal constants; drop top-level example usage."""
    kept, dropped = [], 0
    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.ClassDef)):
            kept.append(node)
        elif isinstance(node, (ast.Assign, ast.AnnAssign)) and node.value is not None:
            try:
                ast.literal_eval(node.value)
                kept.append(node)
            except Exception:
                dropped += 1
        elif isinstance(node, ast.Expr) and isinstance(getattr(node, "value", None), ast.Constant):
            continue                                    # docstring
        else:
            dropped += 1
    return ast.Module(body=kept, type_ignores=[]), dropped


def extract_code(text):
    reason = "no code block with a function"
    for block in candidate_blocks(text):
        try:
            tree = ast.parse(block)
        except SyntaxError:
            reason = "syntax error"
            continue
        funcs = [n.name for n in tree.body if isinstance(n, ast.FunctionDef)]
        if not funcs:
            continue
        if "transform" in funcs:
            name, renamed = "transform", False
        elif len(funcs) == 1:
            name, renamed = funcs[0], True
        else:
            return dict(parse_ok=False, parse_reason=f"several functions, none named transform: {funcs}")
        module, dropped = keep_definitions(tree)
        return dict(parse_ok=True, code=ast.unparse(module), original_block=block, func_name=name,
                    renamed=renamed, top_level_dropped=dropped)
    return dict(parse_ok=False, parse_reason=reason)


ALLOWED_IMPORTS = {"pandas", "numpy", "re", "math", "datetime", "string", "collections", "itertools",
                   "functools", "statistics", "typing"}
BLOCKED_NAMES = {"open", "exec", "eval", "compile", "__import__", "input", "globals", "locals", "vars",
                 "breakpoint", "exit", "quit", "getattr", "setattr", "delattr", "memoryview",
                 "os", "sys", "subprocess", "socket", "shutil", "pathlib", "io", "pickle", "requests",
                 "urllib", "http", "builtins", "importlib", "ctypes", "multiprocessing", "threading",
                 "signal", "glob", "tempfile"}
BLOCKED_ATTRS = {"read_csv", "read_excel", "read_parquet", "read_json", "read_sql", "read_sql_query",
                 "read_sql_table", "read_pickle", "read_table", "read_html", "read_feather", "read_hdf",
                 "read_clipboard", "read_fwf", "read_xml", "to_csv", "to_parquet", "to_pickle", "to_excel",
                 "to_json", "to_sql", "to_hdf", "to_feather", "to_html", "to_xml", "to_clipboard",
                 "HDFStore", "ExcelWriter", "ExcelFile", "load", "save", "savez", "savez_compressed",
                 "savetxt", "loadtxt", "genfromtxt", "fromfile", "tofile", "system", "popen"}


def ast_check(code):
    for node in ast.walk(ast.parse(code)):
        if isinstance(node, ast.Import):
            for a in node.names:
                if a.name.split(".")[0] not in ALLOWED_IMPORTS:
                    return False, f"import {a.name}"
        elif isinstance(node, ast.ImportFrom):
            if (node.module or "").split(".")[0] not in ALLOWED_IMPORTS:
                return False, f"from {node.module} import ..."
        elif isinstance(node, ast.Name) and node.id in BLOCKED_NAMES:
            return False, f"name '{node.id}'"
        elif isinstance(node, ast.Attribute):
            if node.attr.startswith("__") and node.attr.endswith("__"):
                return False, f"dunder attribute '{node.attr}'"
            if node.attr in BLOCKED_ATTRS:
                return False, f"attribute '{node.attr}'"
    return True, ""


print(extract_code("Here you go:\n```python\nimport pandas as pd\ndef transform(df):\n    return df\n\ndf = pd.read_csv('x.csv')\nprint(transform(df))\n```"))
print(ast_check("import os\ndef transform(df):\n    return df"))

{'parse_ok': True, 'code': 'import pandas as pd\n\ndef transform(df):\n    return df', 'original_block': "import pandas as pd\ndef transform(df):\n    return df\n\ndf = pd.read_csv('x.csv')\nprint(transform(df))\n", 'func_name': 'transform', 'renamed': False, 'top_level_dropped': 2}
(False, 'import os')


In [10]:
# ---------------- Execute every candidate in the sandbox and label it ----------------
CAND_PATH = OUT / "candidates.jsonl"
JOBS = OUT / "jobs"
SANDBOX = OUT / "sandbox"
JOBS.mkdir(exist_ok=True)
SANDBOX.mkdir(exist_ok=True)
TASK_BY_ID = {t["id"]: t for t in ALL_TASKS}


def evaluate(gen):
    task = TASK_BY_ID[gen["task_id"]]
    rec = {k: v for k, v in gen.items() if k != "raw_output"}
    rec.update(raw_output=gen["raw_output"], parse_ok=False, blocked=False, ran_ok=False, correct=False,
               status=None, error=None, reason=None)
    ex = extract_code(gen["raw_output"])
    rec.update({k: v for k, v in ex.items() if k != "original_block"})
    if not ex["parse_ok"]:
        rec["status"] = "parse_error"
        return rec
    rec["code_hash"] = hashlib.sha256(ex["code"].encode()).hexdigest()[:16]
    ok, why = ast_check(ex["code"])
    if not ok:
        rec.update(status="blocked", blocked=True, error=why)
        return rec
    job, res = JOBS / f"{gen['gen_id']}.json", JOBS / f"{gen['gen_id']}.result.json"
    job.write_text(json.dumps(dict(task_id=task["id"], data_file=task["data_file"], code=ex["code"], func_name=ex["func_name"])))
    cwd = SANDBOX / gen["gen_id"]
    cwd.mkdir(exist_ok=True)
    try:
        p = harness("run", ROOT, REF_DIR, job, res, CPU_LIMIT_S, MEM_LIMIT_GB, timeout=EXEC_TIMEOUT_S, cwd=cwd)
    except subprocess.TimeoutExpired:
        rec.update(status="timeout", error=f"no result after {EXEC_TIMEOUT_S}s")
        return rec
    if not res.exists():
        rec.update(status="killed", error=f"sandbox exited with code {p.returncode} (CPU/memory limit?) {p.stderr[-300:]}")
        return rec
    rec.update(json.loads(res.read_text()))
    return rec


gens = [g for g in load_jsonl(GEN_PATH) if g["task_id"] in {t["id"] for t in TASKS}]
evaluated = {r["gen_id"] for r in load_jsonl(CAND_PATH)}
todo = [g for g in gens if g["gen_id"] not in evaluated]
print(f"{len(todo)} candidates to evaluate ({len(evaluated)} already done)")
t0 = time.time()
with ThreadPoolExecutor(EVAL_WORKERS) as pool:
    for i, rec in enumerate(pool.map(evaluate, todo), 1):
        rec["accepted_baseline1"] = bool(rec["ran_ok"])       # Baseline 1 accepts anything that runs
        append_jsonl(CAND_PATH, [rec])
        if i % 25 == 0 or i == len(todo):
            print(f"  {i}/{len(todo)}  ({time.time() - t0:.0f}s)")

360 candidates to evaluate (0 already done)
  25/360  (8s)
  50/360  (15s)
  75/360  (23s)
  100/360  (30s)
  125/360  (38s)
  150/360  (45s)
  175/360  (53s)
  200/360  (60s)
  225/360  (68s)
  250/360  (76s)
  275/360  (83s)
  300/360  (90s)
  325/360  (98s)
  350/360  (106s)
  360/360  (108s)


In [11]:
# ---------------- Metrics ----------------
import pandas as pdk   # the kernel's pandas: only used for tables here


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p_ = k / n
    d = 1 + z * z / n
    c = p_ + z * z / (2 * n)
    m = z * math.sqrt(p_ * (1 - p_) / n + z * z / (4 * n * n))
    return (round((c - m) / d, 3), round((c + m) / d, 3))


CRASH = {"error", "bad_return", "timeout", "killed", "memory"}


def summarize(df):
    n = len(df)
    ran = int(df["ran_ok"].sum())
    correct = int(df["correct"].sum())
    silent = int((df["ran_ok"] & ~df["correct"]).sum())
    return dict(n=n, parse_error=int((df["status"] == "parse_error").sum()), blocked=int(df["blocked"].sum()),
                crashed=int(df["status"].isin(CRASH).sum()), ran_ok=ran, correct=correct,
                accuracy=round(correct / n, 3) if n else None, accuracy_ci=wilson(correct, n),
                silent_errors=silent, silent_error_rate=round(silent / ran, 3) if ran else None,
                silent_error_rate_ci=wilson(silent, ran))


C = pdk.DataFrame(load_jsonl(CAND_PATH))
C = C[C["task_id"].isin({t["id"] for t in TASKS})]
for col in ["ran_ok", "correct", "blocked"]:
    C[col] = C[col].fillna(False).astype(bool)

metrics = {}
for split in sorted(C["split"].unique()):
    S = C[C["split"] == split]
    G = S[S["mode"] == "greedy"]
    U = S[S["ran_ok"]].drop_duplicates(["task_id", "code_hash"])
    metrics[split] = dict(
        greedy=summarize(G),
        greedy_by_type={k: summarize(v) for k, v in G.groupby("transformation_type")},
        greedy_by_dataset={k: summarize(v) for k, v in G.groupby("dataset")},
        all_samples=summarize(S),
        all_by_type={k: summarize(v) for k, v in S.groupby("transformation_type")},
        pool_unique_runnable=len(U),
        pool_unique_wrong_by_type=U[~U["correct"]].groupby("transformation_type").size().to_dict(),
        wrong_reasons=S[S["ran_ok"] & ~S["correct"]]["reason"].value_counts().to_dict(),
        renamed_function=int(S.get("renamed", pdk.Series(dtype=bool)).fillna(False).sum()),
        truncated=int(S["truncated"].sum()),
        index_not_kept=int((S["ran_ok"] & (S.get("index_kept") == False)).sum()),
    )
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2, default=str))


def table(d):
    return pdk.DataFrame(d).T[["n", "ran_ok", "correct", "accuracy", "accuracy_ci", "silent_errors",
                               "silent_error_rate", "silent_error_rate_ci", "crashed", "blocked", "parse_error"]]


for split, m in metrics.items():
    print(f"\n========== split = {split} ==========")
    print("Baseline 1 (greedy):")
    display(table({"overall": m["greedy"], **m["greedy_by_type"], **m["greedy_by_dataset"]}))
    print("All candidates (greedy + samples):")
    display(table({"overall": m["all_samples"], **m["all_by_type"]}))
    print("Gate 1 - unique runnable wrong candidates by type:", m["pool_unique_wrong_by_type"],
          "| unique runnable candidates:", m["pool_unique_runnable"])
    print("Why wrong candidates are wrong:", m["wrong_reasons"])
    print(f"renamed function: {m['renamed_function']} | truncated: {m['truncated']} | index not kept: {m['index_not_kept']}")


========== split = dev ==========
Baseline 1 (greedy):


,n,ran_ok,correct,accuracy,accuracy_ci,silent_errors,silent_error_rate,silent_error_rate_ci,crashed,blocked,parse_error
overall,10,10,8,0.8,"(0.49, 0.943)",2,0.2,"(0.057, 0.51)",0,0,0
column_transformation,2,2,2,1.0,"(0.342, 1.0)",0,0.0,"(0.0, 0.658)",0,0,0
deduplication,2,2,1,0.5,"(0.095, 0.905)",1,0.5,"(0.095, 0.905)",0,0,0
filtering,2,2,1,0.5,"(0.095, 0.905)",1,0.5,"(0.095, 0.905)",0,0,0
missing_values,2,2,2,1.0,"(0.342, 1.0)",0,0.0,"(0.0, 0.658)",0,0,0
type_conversion,2,2,2,1.0,"(0.342, 1.0)",0,0.0,"(0.0, 0.658)",0,0,0
adult,5,5,5,1.0,"(0.566, 1.0)",0,0.0,"(-0.0, 0.434)",0,0,0
airbnb,5,5,3,0.6,"(0.231, 0.882)",2,0.4,"(0.118, 0.769)",0,0,0


All candidates (greedy + samples):


,n,ran_ok,correct,accuracy,accuracy_ci,silent_errors,silent_error_rate,silent_error_rate_ci,crashed,blocked,parse_error
overall,60,56,38,0.633,"(0.507, 0.744)",18,0.321,"(0.214, 0.452)",4,0,0
column_transformation,12,12,12,1.0,"(0.757, 1.0)",0,0.0,"(0.0, 0.243)",0,0,0
deduplication,12,11,2,0.167,"(0.047, 0.448)",9,0.818,"(0.523, 0.949)",1,0,0
filtering,12,9,2,0.167,"(0.047, 0.448)",7,0.778,"(0.453, 0.937)",3,0,0
missing_values,12,12,12,1.0,"(0.757, 1.0)",0,0.0,"(0.0, 0.243)",0,0,0
type_conversion,12,12,10,0.833,"(0.552, 0.953)",2,0.167,"(0.047, 0.448)",0,0,0


Gate 1 - unique runnable wrong candidates by type: {'deduplication': 8, 'filtering': 7, 'type_conversion': 2} | unique runnable candidates: 32
Why wrong candidates are wrong: {'row count differs (945 vs 950)': 3, 'row count differs (279 vs 987)': 2, 'values differ': 2, 'dtype family differs in income (int vs string)': 1, 'row count differs (993 vs 989)': 1, 'row count differs (1000 vs 989)': 1, 'row count differs (230 vs 969)': 1, 'row count differs (280 vs 969)': 1, "columns differ (extra=['listing_id'], missing=['id'])": 1, 'row count differs (906 vs 950)': 1, "columns differ (extra=['is_first_instance'], missing=[])": 1, 'row count differs (282 vs 987)': 1, 'row count differs (441 vs 987)': 1, 'row count differs (162 vs 987)': 1}
renamed function: 0 | truncated: 0 | index not kept: 0

========== split = test ==========
Baseline 1 (greedy):


,n,ran_ok,correct,accuracy,accuracy_ci,silent_errors,silent_error_rate,silent_error_rate_ci,crashed,blocked,parse_error
overall,50,48,41,0.82,"(0.692, 0.902)",7,0.146,"(0.072, 0.272)",2,0,0
column_transformation,10,10,10,1.0,"(0.722, 1.0)",0,0.0,"(-0.0, 0.278)",0,0,0
deduplication,10,8,4,0.4,"(0.168, 0.687)",4,0.5,"(0.215, 0.785)",2,0,0
filtering,10,10,9,0.9,"(0.596, 0.982)",1,0.1,"(0.018, 0.404)",0,0,0
missing_values,10,10,10,1.0,"(0.722, 1.0)",0,0.0,"(-0.0, 0.278)",0,0,0
type_conversion,10,10,8,0.8,"(0.49, 0.943)",2,0.2,"(0.057, 0.51)",0,0,0
adult,25,24,20,0.8,"(0.609, 0.911)",4,0.167,"(0.067, 0.359)",1,0,0
airbnb,25,24,21,0.84,"(0.653, 0.936)",3,0.125,"(0.043, 0.31)",1,0,0


All candidates (greedy + samples):


,n,ran_ok,correct,accuracy,accuracy_ci,silent_errors,silent_error_rate,silent_error_rate_ci,crashed,blocked,parse_error
overall,300,284,244,0.813,"(0.765, 0.853)",40,0.141,"(0.105, 0.186)",16,0,0
column_transformation,60,60,60,1.0,"(0.94, 1.0)",0,0.0,"(-0.0, 0.06)",0,0,0
deduplication,60,49,22,0.367,"(0.256, 0.493)",27,0.551,"(0.413, 0.681)",11,0,0
filtering,60,58,53,0.883,"(0.778, 0.942)",5,0.086,"(0.037, 0.186)",2,0,0
missing_values,60,60,60,1.0,"(0.94, 1.0)",0,0.0,"(-0.0, 0.06)",0,0,0
type_conversion,60,57,49,0.817,"(0.701, 0.894)",8,0.14,"(0.073, 0.253)",3,0,0


Gate 1 - unique runnable wrong candidates by type: {'deduplication': 22, 'filtering': 4, 'type_conversion': 6} | unique runnable candidates: 113
Why wrong candidates are wrong: {'values differ': 10, 'row count differs (661 vs 843)': 4, 'dtype family differs in income (int vs category)': 3, 'category order differs in education': 3, 'row count differs (989 vs 986)': 2, "columns differ (extra=['first_occurrence'], missing=[])": 2, "columns differ (extra=['last_row'], missing=[])": 2, "columns differ (extra=['income_num'], missing=['income'])": 1, 'row count differs (999 vs 868)': 1, 'row count differs (1993 vs 993)': 1, 'row count differs (884 vs 991)': 1, 'row count differs (965 vs 991)': 1, 'row count differs (0 vs 993)': 1, 'row count differs (1000 vs 986)': 1, 'dtype family differs in calculated_host_listings_count (float vs int)': 1, 'row count differs (661 vs 868)': 1, "columns differ (extra=['name_host_unique'], missing=[])": 1, 'row count differs (132 vs 868)': 1, "columns differ 

In [12]:
# ---------------- Look at some silent errors ----------------
W = C[C["ran_ok"] & ~C["correct"]].sort_values(["transformation_type", "task_id"])
for _, r in W.drop_duplicates("task_id").head(10).iterrows():
    print("=" * 100)
    print(f"{r['task_id']} [{r['mode']}{r['sample_idx']}]  ->  {r['reason']}")
    print("REQUEST:", TASK_BY_ID[r["task_id"]]["request"])
    print(r["code"])

adult_dedup_001 [sample3]  ->  row count differs (0 vs 993)
REQUEST: Remove duplicate records: two records are duplicates if they match in every column except fnlwgt. Keep the first one.
def transform(df):
    df_no_duplicates = df.loc[df.duplicated(subset=df.columns[:-1], keep='first'), :]
    return df_no_duplicates
adult_dedup_002 [sample0]  ->  values differ
REQUEST: Records that match in every column except fnlwgt are duplicates. Keep only the last occurrence of each.
def transform(df):
    df_sorted = df.sort_values(by='fnlwgt', ascending=False).drop_duplicates(subset=df.columns.difference(['fnlwgt']), keep='last')
    return df_sorted
adult_dedup_003 [greedy0]  ->  values differ
REQUEST: fnlwgt should identify a record, so remove repeated fnlwgt values and keep the first record for each.
def transform(df):
    df = df.sort_values(by='fnlwgt').drop_duplicates(subset='fnlwgt', keep='first')
    return df
adult_dedup_006 [sample0]  ->  dtype family differs in income (int vs string)

In [13]:
# ---------------- Save environment info and zip the outputs ----------------
env = dict(model=MODEL_ID, prompt_version=PROMPT_VERSION, prompt_hash=PROMPT_HASH, sandbox_versions=EVAL_VERSIONS,
           kernel_python=sys.version, n_samples=N_SAMPLES, temperature=TEMPERATURE, top_p=TOP_P,
           max_new_tokens=MAX_NEW_TOKENS, seed=SEED, splits=SPLITS, limit_tasks=LIMIT_TASKS,
           exec_timeout_s=EXEC_TIMEOUT_S, cpu_limit_s=CPU_LIMIT_S, mem_limit_gb=MEM_LIMIT_GB)
try:
    import torch, transformers
    env.update(torch=torch.__version__, transformers=transformers.__version__,
               gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
except Exception:
    pass
(OUT / "env.json").write_text(json.dumps(env, indent=2))
shutil.rmtree(SANDBOX, ignore_errors=True)
archive = shutil.make_archive(str(WORK / "baseline1_outputs"), "zip", OUT)
print("saved:", archive)
print("key files: generations.jsonl, candidates.jsonl (candidate pool v1), metrics.json, env.json")

saved: /kaggle/working/baseline1_outputs.zip
key files: generations.jsonl, candidates.jsonl (candidate pool v1), metrics.json, env.json
